<a href="https://colab.research.google.com/github/nausheen180706-nn/smart-city-prediction/blob/main/garbage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Garbage level detection YOLOV8 model



## lib installations

> Add blockquote



In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from roboflow import Roboflow
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import os

# dataset download

> roboflow



In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="VG50ENAhbdKeIuVGxrsO")

workspace = rf.workspace("sams-workspace-p5asq")

project = workspace.project("waste-bin-fill-level-detect2-lxsf4-a4hw5")

dataset = project.version(1).download("yolov8")

# create yaml files

In [ ]:
for root, dirs, files in os.walk(dataset.location):
    print(root)

In [ ]:
yaml_path = os.path.join(dataset.location, "data.yaml")

with open(yaml_path, "r") as f:
    print(f.read())

# **import yoloV8 model , weight=s**

In [ ]:
model = YOLO("yolov8n.pt")

# **train model with 10-epoches**

In [ ]:
results = model.train(

    data=os.path.join(dataset.location, "data.yaml"),

    epochs=10,

    imgsz=640,

    batch=16,

    workers=2,

    device=0
)

# **Show the metrics**

In [ ]:
metrics = model.val()

In [ ]:
print("Precision :", metrics.box.mp)
print("Recall :", metrics.box.mr)
print("mAP@50 :", metrics.box.map50)
print("mAP@50-95 :", metrics.box.map)

## **Visualization**

In [ ]:
from IPython.display import Image

Image("/content/runs/detect/train/results.png")

In [ ]:
Image("/content/runs/detect/train/confusion_matrix.png")

# **Model's prediction**

In [ ]:
results = model.predict(

    source=os.path.join(dataset.location, "test/images"),

    conf=0.25,

    save=True
)

In [ ]:
prediction_folder = "/content/runs/detect/predict"

files = os.listdir(prediction_folder)

Image(os.path.join(prediction_folder, files[2]))

# **Save model**

In [ ]:
best_model = "/content/runs/detect/train/weights/best.pt"

print(best_model)

In [ ]:
model.export(format="onnx")

In [ ]:
import pickle

with open("garbage_fill_level_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully!")

# **Model tuning**

In [ ]:
test_image = os.path.join(dataset.location, "test/images")

image_name = os.listdir(test_image)[0]

result = model.predict(

    source=os.path.join(test_image, image_name),

    conf=0.25,

    save=True
)

# **replace yolov8 nano wieght model with yolov8 small weight**

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

# again re-creating the yaml file

In [ ]:
import os

yaml_file = os.path.join(dataset.location, "data.yaml")

print(yaml_file)

# **Train model **

In [ ]:
results = model.train(
    data=yaml_file,
    epochs=10,
    imgsz=640,
    batch=16,
    patience=30,
    device=0,
    workers=2
)

In [ ]:
results = model.train(
    data=yaml_file,
    epochs=100,
    imgsz=640,
    batch=16,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2,

    flipud=0.2,
    fliplr=0.5,

    mosaic=1.0,
    mixup=0.2,

    device=0
)

# **Ensembling** **techniques**

In [ ]:
model1 = YOLO("yolov8n.pt")

model1.train(
    data=yaml_file,
    epochs=50,
    imgsz=640
)

In [ ]:
model2 = YOLO("yolov8s.pt")

model2.train(
    data=yaml_file,
    epochs=50,
    imgsz=640
)

In [ ]:
model3 = YOLO("yolov8m.pt")

model3.train(
    data=yaml_file,
    epochs=50,
    imgsz=640
)

# **Test Time Augmentation (TTA)**

In [ ]:
!pip install ensemble-boxes

In [ ]:
results = model.predict(
    source="test.jpg",
    augment=True,
    conf=0.25
)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")   # or your best.pt
print(model)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train/weights/best.pt")

# **Hyperparameter Evolution**

In [ ]:
model.train(
    data=yaml_file,
    epochs=50,

)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train-4/weights/best.pt")   # Change path if needed

metrics = model.val(
    data="/content/waste-bin-fill-level-detect2-1/data.yaml"
)

In [ ]:
print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP@50    :", metrics.box.map50)
print("mAP@50-95 :", metrics.box.map)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train-4/weights/best.pt")

In [ ]:
metrics = model.val(
    data="/content/waste-bin-fill-level-detect2-1/data.yaml"
)

In [ ]:
print("="*50)
print("YOLOv8 Model Evaluation")
print("="*50)

print(f"Precision (mP): {metrics.box.mp:.4f}")
print(f"Recall (mR): {metrics.box.mr:.4f}")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

In [ ]:
print("Per-Class AP:")

for i, ap in enumerate(metrics.box.maps):
    print(f"Class {i}: AP50-95 = {ap:.4f}")

In [ ]:
from IPython.display import Image

Image("/content/runs/detect/train/confusion_matrix.png")

In [ ]:
Image("/content/runs/detect/train/results.png")

In [ ]:
Image("/content/runs/detect/train/PR_curve.png")

In [ ]:
Image("/content/runs/detect/train/P_curve.png")

In [ ]:
Image("/content/runs/detect/train/R_curve.png")

In [ ]:
Image("/content/runs/detect/train/F1_curve.png")

In [ ]:
import os

print(os.listdir("/content/runs/detect"))

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train-4/weights/best.pt")

In [ ]:
filename = list(uploaded.keys())[0]

results = model.predict(
    source=filename,
    conf=0.25,
    save=True,
    show_labels=True,
    show_conf=True
)

In [ ]:
from IPython.display import Image
import glob

pred = glob.glob("/content/runs/detect/predict-3/*.jpg")

Image(pred[0])

In [ ]:
model.train(
    data=yaml_file,
    epochs=100,

)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train-5/weights/best.pt")   # Change path if needed

metrics = model.val(
    data="/content/waste-bin-fill-level-detect2-1/data.yaml"
)

In [ ]:
print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP@50    :", metrics.box.map50)
print("mAP@50-95 :", metrics.box.map)

In [ ]:
from google.colab import files

uploaded=files.upload()

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train-5/weights/best.pt")

In [ ]:
filename = list(uploaded.keys())[0]

results = model.predict(
    source=filename,
    conf=0.25,
    save=True,
    show_labels=True,
    show_conf=True
)

In [ ]:
from IPython.display import Image
import glob

pred = glob.glob("/content/waste-bin-fill-level-detect2-1/test/images/IMG_20251014_170423_jpg.rf.2b309124f46f75fad43f133e3ad26ec1.jpg")
Image(pred[0])